In [1]:
import sys; sys.path.append("..")
import pandas as pd, numpy as np, warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from src.features import get_splits
from src.evaluation import cv_report, to_table

X_train, X_test, y_train, y_test = get_splits("../data/raw/telco_churn.csv")

In [2]:
rows = [cv_report("LogReg C=0.1", LogisticRegression(C=0.1, max_iter=2000), X_train, y_train)]
for d in [2, 3, 4, 5, 6, 8, 12]:
    rows.append(cv_report(f"Tree depth={d}",
                DecisionTreeClassifier(max_depth=d, min_samples_leaf=20, random_state=42),
                X_train, y_train))
tree_table = to_table(rows); tree_table

,model,train_AUC,CV_AUC,CV_AUC_std,CV_PR_AUC,CV_PR_std
0,LogReg C=0.1,0.8503,0.8464,0.0112,0.6630,0.0151
1,Tree depth=2,0.7921,0.7913,0.0072,0.4940,0.0129
2,Tree depth=3,0.8247,0.8166,0.0063,0.5651,0.0132
3,Tree depth=4,0.8414,0.8274,0.0086,0.6036,0.0173
4,Tree depth=5,0.8534,0.8300,0.0096,0.6180,0.0206
5,Tree depth=6,0.8668,0.8253,0.0086,0.6221,0.0192
6,Tree depth=8,0.8844,0.8131,0.0066,0.6115,0.0172
7,Tree depth=12,0.8906,0.8102,0.0086,0.6104,0.0211


In [3]:
rows = []
for d in [4, 6, 8, None]:
    rows.append(cv_report(f"RF depth={d}",
                RandomForestClassifier(n_estimators=300, max_depth=d, min_samples_leaf=5,
                                       n_jobs=-1, random_state=42), X_train, y_train))
rf_table = to_table(rows); rf_table

,model,train_AUC,CV_AUC,CV_AUC_std,CV_PR_AUC,CV_PR_std
0,RF depth=4,0.8490,0.8409,0.0092,0.6459,0.0119
1,RF depth=6,0.8687,0.8460,0.0087,0.6579,0.0124
2,RF depth=8,0.8944,0.8465,0.0084,0.6612,0.0153
3,RF depth=None,0.9261,0.8438,0.0076,0.6540,0.0166


In [4]:
rows = []
for d, lr, n in [(2, 0.05, 300), (3, 0.05, 300), (3, 0.05, 150), (4, 0.05, 200)]:
    rows.append(cv_report(f"XGB d={d} lr={lr} n={n}",
                XGBClassifier(n_estimators=n, max_depth=d, learning_rate=lr,
                              subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
                              tree_method="hist", random_state=42, n_jobs=1),
                X_train, y_train))
xgb_table = to_table(rows); xgb_table

,model,train_AUC,CV_AUC,CV_AUC_std,CV_PR_AUC,CV_PR_std
0,XGB d=2 lr=0.05 n=300,0.8679,0.8491,0.0120,0.6700,0.0203
1,XGB d=3 lr=0.05 n=300,0.8877,0.8465,0.0106,0.6650,0.0173
2,XGB d=3 lr=0.05 n=150,0.8732,0.8495,0.0111,0.6692,0.0202
3,XGB d=4 lr=0.05 n=200,0.9008,0.8461,0.0106,0.6655,0.0218
